# 🎯 Notebook 05 — Downstream Tasks
## Financial Representation Learning System

**Purpose:** Demonstrate the value of the enriched customer embeddings through four downstream applications:

1. **AEPD Segmentation** — K-Means clustering → customer journey stage
2. **UMAP Visualisation** — 2D projection for stakeholder reporting  
3. **Anomaly Detection** — flag outlier customers for review
4. **Next-Product Prediction** — MLP classifier on embeddings

This is the business-facing output. One embedding. Four use cases.

**Runtime:** ~5 minutes (no GPU required)

## Setup

In [ ]:
import os, sys
os.chdir("/content/frl-system")
sys.path.insert(0, ".")

# Install UMAP if needed
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "umap-learn", "-q"])

import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.downstream import (
    segment_customers, compute_umap, plot_umap,
    detect_anomalies, train_next_product_classifier,
    generate_segment_profiles
)

print("✅ Imports ready")

## Step 1 — Load Data

In [ ]:
with open("data/synthetic/enriched_embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

transactions = pd.read_csv("data/synthetic/transactions.csv")
customers    = pd.read_csv("data/synthetic/customers.csv")
products     = pd.read_csv("data/synthetic/products.csv")

print(f"✅ Loaded {len(embeddings):,} enriched customer embeddings")

## Task 1 — AEPD Segmentation

In [ ]:
meta, kmeans = segment_customers(embeddings, n_segments=4)
print("\nSegmentation complete.")
print(meta[["customer_id", "aepd_pred", "aepd_true", "archetype", "centroid_distance"]].head(10))

## Task 2 — UMAP Visualisation

In [ ]:
coords = compute_umap(embeddings)

if coords is not None:
    plot_umap(coords, meta,
              title="Customer Embedding Space — FRL System",
              save_path="docs/results/05_umap.png")

## Task 3 — Anomaly Detection

In [ ]:
meta = detect_anomalies(meta, threshold_percentile=95)

# Show top anomalies
print("\nTop 10 Anomalous Customers:")
top_anomalies = meta[meta['is_anomaly']].nlargest(10, 'centroid_distance')
print(top_anomalies[["customer_id", "aepd_pred", "archetype", "centroid_distance"]].to_string(index=False))

## Task 4 — Next-Product Prediction

In [ ]:
results = train_next_product_classifier(embeddings, products, product_type="PERSONAL_LOAN")
accuracy = results['report']['accuracy']
print(f"\nClassifier Accuracy: {accuracy:.3f}")

## Task 5 — Segment Profile Report

In [ ]:
profiles = generate_segment_profiles(meta, transactions, products)

## Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("FRL System — Results Dashboard", fontsize=16, fontweight='bold')

aepd_colors = {'A': '#E74C3C', 'E': '#F39C12', 'P': '#2ECC71', 'D': '#3498DB'}

# 1. Segment distribution
seg_counts = meta['aepd_pred'].value_counts()
bars = axes[0,0].bar(seg_counts.index, seg_counts.values,
                      color=[aepd_colors.get(s,'#999') for s in seg_counts.index])
axes[0,0].set_title("AEPD Segment Distribution", fontweight='bold')
axes[0,0].set_xlabel("AEPD Stage")
axes[0,0].set_ylabel("Number of Customers")
for bar in bars:
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                   f'{int(bar.get_height()):,}', ha='center', fontsize=10)

# 2. Ground truth vs predicted
from sklearn.metrics import confusion_matrix
import seaborn as sns
valid = meta[(meta['aepd_true'] != 'UNKNOWN') & (meta['aepd_pred'].notna())]
stages = sorted(valid['aepd_true'].unique())
cm = confusion_matrix(valid['aepd_true'], valid['aepd_pred'], labels=stages)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,1],
            xticklabels=stages, yticklabels=stages)
axes[0,1].set_title("Ground Truth vs Predicted AEPD", fontweight='bold')
axes[0,1].set_xlabel("Predicted")
axes[0,1].set_ylabel("True")

# 3. Anomaly distribution
anomaly_counts = meta['is_anomaly'].value_counts()
axes[1,0].pie([anomaly_counts.get(False,0), anomaly_counts.get(True,0)],
              labels=['Normal', 'Anomaly'],
              colors=['#2ECC71', '#E74C3C'],
              autopct='%1.1f%%', startangle=90)
axes[1,0].set_title("Customer Anomaly Detection", fontweight='bold')

# 4. Centroid distance distribution by segment
for stage in ['A','E','P','D']:
    mask = meta['aepd_pred'] == stage
    if mask.sum() > 0:
        axes[1,1].hist(meta[mask]['centroid_distance'], bins=30, alpha=0.6,
                       label=stage, color=aepd_colors[stage])
axes[1,1].set_xlabel("Distance from Cluster Centroid")
axes[1,1].set_ylabel("Count")
axes[1,1].set_title("Cluster Cohesion by AEPD Stage", fontweight='bold')
axes[1,1].legend()

plt.tight_layout()
plt.savefig("docs/results/05_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Dashboard saved to docs/results/05_dashboard.png")

## 🎉 End-to-End MVP Complete!

### What the FRL System Demonstrated:

| Component | Status | Output |
|---|---|---|
| Synthetic Data Generator | ✅ | 313,000 rows across 6 tables |
| Event Tokenizer | ✅ | 5,000 × [256 × 8] token sequences |
| CoLES Sequence Encoder | ✅ | 5,000 × 64-dim customer embeddings |
| Bipartite Graph Builder | ✅ | Customer–merchant network |
| GraphSAGE Enrichment | ✅ | Network-aware enriched embeddings |
| AEPD Segmentation | ✅ | 4-stage customer journey classification |
| UMAP Visualisation | ✅ | 2D stakeholder-ready projection |
| Anomaly Detection | ✅ | Top 5% flagged for review |
| Next-Product Prediction | ✅ | MLP on embeddings |

### Key Results:
- **One embedding, multiple applications** — the same 64-dim vector powers all 4 downstream tasks
- **No labels required** — CoLES and GraphSAGE are both self-supervised
- **Inductive** — GraphSAGE works on new customers not seen during training

### Production Next Steps:
- Scale to full customer base (Microsoft Fabric + Azure ML)
- Swap GRU encoder for Transformer (BST-style) for higher accuracy
- Add real-time embedding updates via event streams
- Integrate with CRM for campaign targeting

See `production/fabric/README.md` for the full production deployment guide.